# 01 · Bronze — dado cru, com rastreabilidade

```
landing (arquivos crus) ──► BRONZE (Iceberg) ──► Silver ──► Gold ─┬─► Postgres ──► Metabase
                                                                  └─► chunks ──► pgvector (próximo módulo)
```

**Regra da Bronze:** guardar como veio. Nada de limpar aqui. Só acrescentamos 3 colunas para saber *de onde* e *quando* veio:
`_ingest_ts`, `_source_file`, `_batch_id`.

**Antes de começar:** `docker compose up -d` (pasta `docker/`) e `pip install -r requirements.txt`.

## Passo 0 · O ambiente está de pé?

In [1]:
import pyspark
from pathlib import Path

pasta = Path(pyspark.__file__).parent

print("PySpark:", pyspark.__version__)

for jar in (pasta / "jars").glob("spark-core_*.jar"):
    print("Spark Core:", jar.name)

PySpark: 3.5.9
Spark Core: spark-core_2.12-3.5.3.jar
Spark Core: spark-core_2.12-3.5.9.jar


In [ ]:
from pathlib import Path
import pyspark

pasta_jars = Path(pyspark.__file__).parent / "jars"

jar_antigo = pasta_jars / "spark-core_2.12-3.5.3.jar"

if jar_antigo.exists():
    jar_antigo.unlink()
    print("JAR antigo removido:", jar_antigo.name)
else:
    print("JAR antigo não encontrado.")

In [2]:
import os
import sys
from pathlib import Path

# Forçar o Python do Jupyter como Python do Spark
PYTHON_EXE = sys.executable

os.environ["PYSPARK_PYTHON"] = PYTHON_EXE
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_EXE

# Caminho do projeto
ROOT = Path(
    r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao"
)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python definido:", PYTHON_EXE)
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

Python definido: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
PYSPARK_PYTHON: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
SPARK_HOME: None


In [3]:
import pyspark
from pathlib import Path

pasta_jars = Path(pyspark.__file__).parent / "jars"

for jar in sorted(pasta_jars.glob("spark-*.jar")):
    print(jar.name)

spark-catalyst_2.12-3.5.3.jar
spark-catalyst_2.12-3.5.9.jar
spark-common-utils_2.12-3.5.3.jar
spark-common-utils_2.12-3.5.9.jar
spark-core_2.12-3.5.3.jar
spark-core_2.12-3.5.9.jar
spark-graphx_2.12-3.5.3.jar
spark-graphx_2.12-3.5.9.jar
spark-hive-thriftserver_2.12-3.5.3.jar
spark-hive-thriftserver_2.12-3.5.9.jar
spark-hive_2.12-3.5.3.jar
spark-hive_2.12-3.5.9.jar
spark-kubernetes_2.12-3.5.3.jar
spark-kubernetes_2.12-3.5.9.jar
spark-kvstore_2.12-3.5.3.jar
spark-kvstore_2.12-3.5.9.jar
spark-launcher_2.12-3.5.3.jar
spark-launcher_2.12-3.5.9.jar
spark-mesos_2.12-3.5.3.jar
spark-mesos_2.12-3.5.9.jar
spark-mllib-local_2.12-3.5.3.jar
spark-mllib-local_2.12-3.5.9.jar
spark-mllib_2.12-3.5.3.jar
spark-mllib_2.12-3.5.9.jar
spark-network-common_2.12-3.5.3.jar
spark-network-common_2.12-3.5.9.jar
spark-network-shuffle_2.12-3.5.3.jar
spark-network-shuffle_2.12-3.5.9.jar
spark-repl_2.12-3.5.3.jar
spark-repl_2.12-3.5.9.jar
spark-sketch_2.12-3.5.3.jar
spark-sketch_2.12-3.5.9.jar
spark-sql-api_2.12-3.5.3

In [2]:
from common import lake_local as L
import pyspark

print("PySpark instalado:", pyspark.__version__)
print("Local do PySpark:", pyspark.__file__)
print("Pacotes:", L.PACKAGES)

spark = L.get_spark()

print("\nSpark iniciado:", spark.version)
print("Python executado pelo Spark:", spark.sparkContext.pythonExec)

print(
    "Configuração pyspark.python:",
    spark.sparkContext.getConf().get(
        "spark.pyspark.python", "NÃO CONFIGURADO"
    )
)

PySpark instalado: 3.5.9
Local do PySpark: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\pyspark\__init__.py
Pacotes: ['org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1', 'org.apache.hadoop:hadoop-aws:3.3.4', 'org.postgresql:postgresql:42.7.4']

Spark iniciado: 3.5.3
Python executado pelo Spark: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Configuração pyspark.python: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe


In [3]:
resultado = (
    spark.sparkContext
    .parallelize([1, 2, 3], 2)
    .map(lambda x: x * 10)
    .collect()
)

print("Resultado:", resultado)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(I

In [4]:
import sys
import subprocess
from pathlib import Path

print("Executável existe:", Path(sys.executable).exists())
print("Versão:", sys.version)

r = subprocess.run(
    [sys.executable, "-c", "import sys; print(sys.version); print('PYTHON OK')"],
    capture_output=True,
    text=True,
    timeout=15
)

print("Código de saída:", r.returncode)
print("Saída:", r.stdout)
print("Erro:", r.stderr)

Executável existe: True
Versão: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Código de saída: 0
Saída: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PYTHON OK

Erro: 


In [5]:
import pyspark
from pathlib import Path

pasta = Path(pyspark.__file__).parent
pasta_jars = pasta / "jars"

print("PySpark:", pyspark.__version__)
print("Local:", pasta)
print("Pasta JARs existe:", pasta_jars.exists())

for jar in pasta_jars.glob("spark-core_*.jar"):
    print("JAR encontrado:", jar.name)

PySpark: 3.5.9
Local: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\pyspark
Pasta JARs existe: True
JAR encontrado: spark-core_2.12-3.5.3.jar


In [6]:
spark.sparkContext.setLogLevel("INFO")

try:
    resultado = (
        spark.sparkContext
        .parallelize([1, 2, 3], 2)
        .map(lambda x: x * 10)
        .collect()
    )

    print("Resultado:", resultado)

except Exception as e:
    print("\nTIPO:", type(e).__name__)
    print("\nERRO COMPLETO:")
    print(e)


TIPO: Py4JJavaError

ERRO COMPLETO:
An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1.0 (TID 2) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIte

In [2]:
import subprocess

pacotes = [
    "pyspark==3.5.3",
    "jupyterlab",
    "pandas>=2.2.2",
    "pyarrow>=16.1.0",
    "boto3>=1.34.136",
    "psycopg2-binary>=2.9.9",
    "delta-spark==3.2.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", *pacotes
])

NameError: name 'sys' is not defined

In [3]:
import sys
import importlib.metadata as metadata

print("Python utilizado:", sys.executable)

pacotes = [
    "pyspark",
    "jupyterlab",
    "pandas",
    "pyarrow",
    "boto3",
    "psycopg2-binary",
    "fastembed"
]

for pacote in pacotes:
    try:
        print(f"{pacote}: {metadata.version(pacote)}")
    except metadata.PackageNotFoundError:
        print(f"{pacote}: NÃO INSTALADO")

Python utilizado: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
pyspark: NÃO INSTALADO
jupyterlab: 4.6.4
pandas: 3.1.0
pyarrow: 26.0.0
boto3: 1.43.118
psycopg2-binary: 2.9.13
fastembed: NÃO INSTALADO


In [4]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "python-dotenv"
])

0

In [ ]:
import sys, subprocess
r = subprocess.run([sys.executable, "../scripts/verifica_ambiente.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)

## Passo 0.5 · Baixar as dependências do Spark (só na primeira vez)

O Spark precisa de alguns jars (Iceberg, S3A com o AWS SDK, JDBC). Na **primeira execução** ele os baixa da internet, e isso pode levar **vários minutos**, principalmente com a rede da sala dividida. Os arquivos ficam guardados em cache (pasta `.ivy2`) e, depois disso, o Spark abre rápido.

Rode a célula abaixo **em casa, antes da aula**. Ela abre um processo separado, baixa os jars e termina, sem deixar o Spark aberto neste notebook.

In [2]:
import time
import boto3
from botocore.config import Config

inicio = time.perf_counter()

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9001",
    aws_access_key_id="admin",
    aws_secret_access_key="admin12345",
    region_name="us-east-1",
    config=Config(
        connect_timeout=3,
        read_timeout=5,
        retries={"max_attempts": 1}
    )
)

try:
    resposta = s3.list_buckets()
    print("Buckets:", [b["Name"] for b in resposta["Buckets"]])
except Exception as e:
    print("Erro:", repr(e))

print("Tempo:", round(time.perf_counter() - inicio, 2), "segundos")

Buckets: ['landing', 'warehouse']
Tempo: 6.71 segundos


In [3]:
import os
import sys
import time

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

inicio = time.perf_counter()

spark_teste = (
    SparkSession.builder
    .master("local[2]")
    .appName("diagnostico-tempo")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

print("Sessão criada:", round(time.perf_counter() - inicio, 2), "segundos")

inicio_operacao = time.perf_counter()

resultado = spark_teste.sql("SELECT 1 AS teste").collect()

print("Consulta:", resultado)
print("Tempo da consulta:", round(time.perf_counter() - inicio_operacao, 2), "segundos")

Sessão criada: 8.67 segundos
Consulta: [Row(teste=1)]
Tempo da consulta: 3.81 segundos


In [4]:
spark_teste.stop()

In [5]:
import time
from common import lake_local as L

inicio = time.perf_counter()

print("Iniciando L.get_spark()...", flush=True)

spark = L.get_spark()

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Tempo total:", round(time.perf_counter() - inicio, 2), "segundos")

ModuleNotFoundError: No module named 'common'

In [6]:
import os
import sys
import time
from pathlib import Path

# Raiz do projeto
ROOT = Path(r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao")

# Configura o diretório de trabalho
os.chdir(ROOT)

# Adiciona a raiz ao caminho de importação
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Confirma localização
print("Diretório:", os.getcwd())
print("Common existe:", (ROOT / "common").is_dir())

# Importa o módulo
from common import lake_local as L

# Mede inicialização
inicio = time.perf_counter()

print("Iniciando L.get_spark()...", flush=True)

spark = L.get_spark()

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Storage:", L.STORAGE)
print("Endpoint:", L.MINIO_ENDPOINT)
print("Tempo total:", round(time.perf_counter() - inicio, 2), "segundos")

Diretório: E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao
Common existe: True
Iniciando L.get_spark()...


Py4JJavaError: An error occurred while calling o99.sql.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'lake': org.apache.iceberg.spark.SparkCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1926)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:54)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.catalog(CatalogManager.scala:54)
	at org.apache.spark.sql.connector.catalog.LookupCatalog$CatalogAndNamespace$.unapply(LookupCatalog.scala:86)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala:51)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala:30)


In [1]:
import os
import sys
import time
from pathlib import Path

ROOT = Path(r"E:\AULA_POS_ARQUITETURA\lidiane_projeto\meu_projeto_medalhao")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from common import lake_local as L

print("PySpark:", __import__("pyspark").__version__)
print("Pacotes configurados:", L.PACKAGES)
print("Storage:", L.STORAGE)
print("MinIO:", L.MINIO_ENDPOINT)

inicio = time.perf_counter()

try:
    spark = L.get_spark()

    print("Spark:", spark.version)
    print("Master:", spark.sparkContext.master)
    print("Tempo:", round(time.perf_counter() - inicio, 2), "segundos")

except Exception as e:
    print("\nERRO:", repr(e))

    if hasattr(e, "java_exception"):
        print("\nCAUSA JAVA:")
        print(e.java_exception.toString())

PySpark: 3.5.9
Pacotes configurados: ['org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1', 'org.apache.hadoop:hadoop-aws:3.3.4', 'org.postgresql:postgresql:42.7.4']
Storage: minio
MinIO: http://localhost:9001
Spark: 3.5.3
Master: local[2]
Tempo: 21.71 segundos


In [2]:
for ns in ["bronze", "silver", "gold"]:
    print(f"\nNamespace: {ns}")
    spark.sql(f"SHOW NAMESPACES IN lake").show()


Namespace: bronze
+---------+
|namespace|
+---------+
|   bronze|
|     gold|
|   silver|
+---------+


Namespace: silver
+---------+
|namespace|
+---------+
|   bronze|
|     gold|
|   silver|
+---------+


Namespace: gold
+---------+
|namespace|
+---------+
|   bronze|
|     gold|
|   silver|
+---------+



In [3]:
df_teste = spark.createDataFrame(
    [(1, "Bronze"), (2, "Silver"), (3, "Gold")],
    ["id", "camada"]
)

df_teste.writeTo(
    "lake.bronze.teste_inicializacao"
).using("iceberg").createOrReplace()

print("Tabela criada!")

spark.sql("""
    SELECT * 
    FROM lake.bronze.teste_inicializacao
""").show()

Py4JJavaError: An error occurred while calling o100.createOrReplace.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-8IQO9ET executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:203)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:67)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:367)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:331)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadChe

In [4]:
import os
import sys

print("Python do Jupyter:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))

print("Python configurado no Spark:",
      spark.sparkContext.pythonExec)

print("\nTESTE 1 — Execução nativa do Spark")

try:
    spark.range(1, 4).show()
    print("TESTE 1: OK")
except Exception as e:
    print("TESTE 1: ERRO", repr(e))

print("\nTESTE 2 — Execução com Python Worker")

try:
    teste = spark.createDataFrame(
        [(1, "Bronze"), (2, "Silver"), (3, "Gold")],
        ["id", "camada"]
    )
    teste.show()
    print("TESTE 2: OK")
except Exception as e:
    print("TESTE 2: ERRO", repr(e))

Python do Jupyter: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
PYSPARK_PYTHON: None
PYSPARK_DRIVER_PYTHON: None
Python configurado no Spark: python3

TESTE 1 — Execução nativa do Spark
+---+
| id|
+---+
|  1|
|  2|
|  3|
+---+

TESTE 1: OK

TESTE 2 — Execução com Python Worker
TESTE 2: ERRO Py4JJavaError('An error occurred while calling o124.showString.\n', JavaObject id=o125)


In [5]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [6]:
from common import lake_local as L

spark = L.get_spark()

In [7]:
import sys

print("Python:", sys.executable)
print(
    "Python do Spark:",
    spark.sparkContext.getConf().get("spark.pyspark.python")
)

resultado = (
    spark.sparkContext
    .parallelize([1, 2, 3], 2)
    .map(lambda x: x * 10)
    .collect()
)

print("Resultado:", resultado)

Python: c:\Users\Lidiane\AppData\Local\Python\pythoncore-3.12-64\python.exe
Python do Spark: None


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 3.0 failed 1 times, most recent failure: Lost task 1.0 in stage 3.0 (TID 6) (DESKTOP-8IQO9ET executor driver): java.net.SocketException: Connection reset
	at java.base/sun.nio.ch.NioSocketImpl.implRead(NioSocketImpl.java:328)
	at java.base/sun.nio.ch.NioSocketImpl.read(NioSocketImpl.java:355)
	at java.base/sun.nio.ch.NioSocketImpl$1.read(NioSocketImpl.java:808)
	at java.base/java.net.Socket$SocketInputStream.read(Socket.java:966)
	at java.base/java.io.BufferedInputStream.fill(BufferedInputStream.java:244)
	at java.base/java.io.BufferedInputStream.read(BufferedInputStream.java:263)
	at java.base/java.io.DataInputStream.readInt(DataInputStream.java:381)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:774)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Py

In [ ]:
import sys, subprocess

r = subprocess.run([sys.executable, "../scripts/preparar_jars.py"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])      # fim da saída: deve terminar com "Jars baixados e prontos"
print(r.stderr[-1500:])      # se houver erro, a causa aparece aqui

## Passo 1 · Abrir o Spark (modo local)

O Spark roda **dentro deste Python**, com `local[2]` (2 núcleos) e 2 GB. Na primeira vez ele baixa os jars (Iceberg, S3A, JDBC). O S3A traz junto o AWS SDK, que é grande, então pode levar vários minutos e precisa de internet. Os jars ficam em cache em ~/.ivy2 e nas próximas execuções é rápido.
Para mais folga, `SPARK_CORES` e `SPARK_MEMORY` (variáveis de ambiente antes de abrir o Jupyter).
Spark UI: <http://localhost:4040> enquanto houver um job rodando.

In [ ]:
from common import lake_local as L

s3 = L._s3()

# Consultar os buckets existentes
buckets = {
    b["Name"] for b in s3.list_buckets()["Buckets"]
}

print("Buckets existentes:", buckets)

# Criar o bucket warehouse, se necessário
if "warehouse" not in buckets:
    s3.create_bucket(Bucket="warehouse")
    print("Bucket warehouse criado com sucesso!")
else:
    print("O bucket warehouse já existe.")

# Conferir novamente
buckets = {
    b["Name"] for b in s3.list_buckets()["Buckets"]
}

print("Buckets disponíveis:", buckets)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L #, medalhao as M usaremos depois
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

## Passo 2 · Os arquivos chegam na landing (object storage)

In [ ]:
from common import lake_local as L

s3 = L._s3()

# Listar buckets existentes
buckets = {
    b["Name"] for b in s3.list_buckets()["Buckets"]
}
print("Buckets atuais:", buckets)

# Criar os buckets necessários
for nome in ["landing", "warehouse"]:
    if nome not in buckets:
        s3.create_bucket(Bucket=nome)
        print(f"Bucket criado: {nome}")
    else:
        print(f"Bucket já existe: {nome}")

# Conferir novamente
print(
    "Buckets finais:",
    [b["Name"] for b in s3.list_buckets()["Buckets"]]
)

In [ ]:
from common import lake_local as L
from pathlib import Path

print("Raiz do projeto:", L.ROOT)
print("Diretório de dados:", L.DATA_DIR)
print("Pasta procurada:", L.DATA_DIR / "lote1")
print("Pasta existe:", (L.DATA_DIR / "lote1").exists())

In [ ]:
from pathlib import Path
from common import lake_local as L

print("Conteúdo da pasta data:")

if L.DATA_DIR.exists():
    for item in L.DATA_DIR.iterdir():
        tipo = "PASTA" if item.is_dir() else "ARQUIVO"
        print(f"{tipo}: {item.name}")
else:
    print("A pasta data não existe!")

In [ ]:
print("Arquivos CSV encontrados no projeto:")

for arquivo in L.ROOT.rglob("*.csv"):
    print(arquivo)

In [ ]:
from common import lake_local as L

pasta = L.DATA_DIR / "lote1"

print("Pasta:", pasta)
print("Existe:", pasta.exists())

for arquivo in pasta.iterdir():
    if arquivo.is_file():
        print(f"{arquivo.name} - {arquivo.stat().st_size:,} bytes")

In [ ]:
n = L.upload_to_landing(
    L.DATA_DIR / "lote1",
    "mercadonordeste/lote1"
)

print(n, "arquivos enviados")

In [ ]:
arquivos = L.list_landing("mercadonordeste/lote1")

print("Total de arquivos encontrados:", len(arquivos))

for chave, tam in arquivos:
    print(f"{tam:>10,} bytes  {chave}")

In [ ]:
arquivos = L.list_landing("mercadonordeste/lote1")

print("Total de arquivos encontrados:", len(arquivos))

for chave, tam in arquivos:
    print(f"{tam:>10,} bytes  {chave}")

In [ ]:
n = L.upload_to_landing(L.DATA_DIR / "lote1", "mercadonordeste/lote1")
print(n, "arquivos enviados")
for chave, tam in L.list_landing("mercadonordeste/lote1"):
    print(f"{tam:>10,} bytes  {chave}")

Abra o console do MinIO (<http://localhost:9001>, `admin` / `admin12345`) e veja o bucket **landing**.
É um *object storage*: não há pastas de verdade, só chaves com `/` no nome. 
Você achou os objetos no bucket warehouse do MinIO, em lakehouse/bronze/<tabela>/data e metadata.

## Passo 3 · Landing → Bronze

In [ ]:
from pathlib import Path
from common import lake_local as L

print("Raiz do projeto:", L.ROOT)
print("\nPastas e arquivos na raiz:")

for item in L.ROOT.iterdir():
    print(item.name, "(pasta)" if item.is_dir() else "(arquivo)")

print("\nArquivos Python que contêm ingest_lote:")

for arquivo in L.ROOT.rglob("*.py"):
    if ".venv" in arquivo.parts:
        continue

    try:
        conteudo = arquivo.read_text(encoding="utf-8")
        if "def ingest_lote" in conteudo:
            print(arquivo)
    except (UnicodeDecodeError, OSError):
        pass

In [ ]:
from common import medalhao as M
contagens = M.ingest_lote(spark, "lote1", L.landing_uri)
contagens

In [ ]:
from pathlib import Path
from common import lake_local as L

for arquivo in L.ROOT.rglob("*.py"):
    if ".venv" not in str(arquivo):
        try:
            conteudo = arquivo.read_text(encoding="utf-8")
            if "def ingest_lote" in conteudo:
                print("Módulo encontrado:", arquivo)
        except (UnicodeDecodeError, OSError):
            pass

In [ ]:
spark.sql("SHOW TABLES IN lake.bronze").show()
spark.table("lake.bronze.pedidos").printSchema()
spark.table("lake.bronze.pedidos").select("pedido_id","cliente_id","status","itens","_batch_id","_source_file").show(3, truncate=60)

Repare: `itens` chegou como **lista de structs** (o JSON foi preservado) e os CSVs chegaram como texto. A tipagem é problema da Silver.

## Passo 4 · Idempotência
Rode o Passo 3 de novo. As contagens **não** dobram: a ingestão apaga o lote e regrava. Pipeline que duplica ao reexecutar é pipeline frágil.

In [ ]:
antes = spark.table("lake.bronze.pedidos").count()
M.ingest_lote(spark, "lote1", L.landing_uri)
depois = spark.table("lake.bronze.pedidos").count()
print(antes, "->", depois, "| idempotente:", antes == depois)

## Checkpoint
- [ ] `lake.bronze` tem `clientes`, `produtos`, `pedidos`, `avaliacoes`.
- [ ] Reexecutar não duplica.
- [ ] Você achou os objetos no bucket `warehouse` do MinIO (pastas `bronze/…/data` e `metadata`).

➡️ Próximo: `02_silver.ipynb`